# Hamiltonian Time Evolution

## Energy in Quantum systems

In [1]:
import pennylane as qml
from pennylane import numpy as np

### Evolution of two qubits

In [2]:
n_bits = 2
dev = qml.device("default.qubit", wires=n_bits)

@qml.qnode(dev)
def zz_circuit(alpha, time, init):
    """Circuit for evolving two electrons with a ZZ interaction.
    
    Args:
        alpha (float): The strength of the interaction.
        time (float): The time we evolve the electron wavefunction for.
        init (numpy.array(int)): An initial state specified by two bits [x, y]. Prepare the
            system in this state prior to applying the time evolution circuit.

    Returns: 
        array[float]: Probabilities for observing different outcomes.
    """

    ##################
    # YOUR CODE HERE #
    ##################
    # init state
    qml.BasisState(init, wires = range(n_bits))

    # apply circuit
    theta = 2*alpha*time
    qml.CNOT(wires = [0,1])
    qml.RZ(theta, 1)
    qml.CNOT(wires = [0,1])

    return qml.probs(wires=range(n_bits))

alpha, time = 0.1, 1 
init = np.array([0,1])
print(qml.draw(zz_circuit)(alpha, time, init))

0: ─╭|Ψ⟩─╭●───────────╭●─┤ ╭Probs
1: ─╰|Ψ⟩─╰X──RZ(0.20)─╰X─┤ ╰Probs


### An ising gate

In [3]:
n_bits = 2
dev = qml.device("default.qubit", wires=n_bits)

@qml.qnode(dev)
def ising_circuit(alpha, time, init):
    """Circuit for evolving two electrons with a ZZ interaction
    using an Ising gate
    
    Args:
        alpha (float): The strength of the interaction.
        time (float): The time we evolve the electron wavefunction for.
        init (numpy.array(int)): An initial state specified by two bits [x, y]. Prepare the
            system in this state prior to applying the time evolution circuit.

    Returns: 
        np.tensor: Output state.
    """
    ##################
    # YOUR CODE HERE #
    ##################
    qml.BasisState(init, wires = range(n_bits))

    qml.IsingZZ(2*alpha*time, wires = range(n_bits))
    return qml.state()

print(qml.draw(ising_circuit)(alpha, time, init))

0: ─╭|Ψ⟩─╭IsingZZ(0.20)─┤  State
1: ─╰|Ψ⟩─╰IsingZZ(0.20)─┤  State


### Built an evolution

In [4]:
n_bits = 2
dev = qml.device("default.qubit", wires=n_bits)

@qml.qnode(dev)
def ZZ_evolve(alpha, time, init):
    """Circuit for evolving two electrons with a ZZ interaction
    using qml.evolve
    
    Args:
        alpha (float): The strength of the interaction.
        time (float): The time we evolve the electron wave function for.
        init (numpy.array(int)): An initial state specified by two bits [x, y]. Prepare the
            system in this state prior to applying the time evolution circuit.

    Returns: 
        np.tensor: Output state.
    """
    ##################
    # YOUR CODE HERE #
    ##################
    qml.BasisState(init, wires = range(n_bits))

    qml.evolve(alpha*qml.PauliZ(0)@qml.PauliZ(1), coeff = time)
    return qml.state()

print(qml.draw(ZZ_evolve)(alpha, time, init))


0: ─╭|Ψ⟩─╭Exp(-1.00j (0.10*Z)@Z)─┤  State
1: ─╰|Ψ⟩─╰Exp(-1.00j (0.10*Z)@Z)─┤  State


### Building the graph Hamiltonian

In [5]:
n_bits = 5
dev = qml.device("default.qubit", wires=n_bits)
    
##################
# YOUR CODE HERE #
##################
coeffs = [1, 1, 1, 1] # MODIFY THIS
obs = [qml.PauliZ(0)@qml.PauliZ(1), qml.PauliZ(1)@qml.PauliZ(2), qml.PauliZ(1)@qml.PauliZ(3), qml.PauliZ(3)@qml.PauliZ(4)] # MODIFY THIS
H = qml.dot(coeffs, obs)

@qml.qnode(dev)
def energy(init):
    """Circuit for measuring expectation value of Hamiltonian in a given state.
    
    Args:
        init (numpy.array(int)): An initial computational basis state, specified by five bits.

    Returns: 
        float: Expectation value of the Hamiltonian H.
    """
    qml.BasisState(init, wires=range(n_bits))
    return qml.expval(H)

init = [1,0,1,0,0]
print(qml.draw(energy)(init))


0: ─╭|Ψ⟩─┤ ╭<𝓗>
1: ─├|Ψ⟩─┤ ├<𝓗>
2: ─├|Ψ⟩─┤ ├<𝓗>
3: ─├|Ψ⟩─┤ ├<𝓗>
4: ─╰|Ψ⟩─┤ ╰<𝓗>


### Guessing the ground state

In [6]:
my_guess1 = np.array([0,1,0,0,1]) # MODIFY THIS
my_guess2 = np.array([1,0,1,1,0]) # MODIFY THIS

print("The expected energy for", my_guess1, "is", energy(my_guess1), ".")
print("The expected energy for", my_guess2, "is", energy(my_guess2), ".")


The expected energy for [0 1 0 0 1] is -4.0 .
The expected energy for [1 0 1 1 0] is -4.0 .
